# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

# Research Question

Can historical search performance and engagement metrics be used to identify content that should be refreshed?

## Decision Supported

This project supports content managers in prioritizing pages for review by ranking content based on observed search performance trends and engagement metrics.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

# 2. Data

## Dataset

This analysis uses the FlyRank ML Internship Warehouse dataset.

## Table Used

- `fact_content_daily_performance`

## Date Window

The analysis uses data from March 1, 2026 to March 31, 2026.

For modeling:

- Historical feature window: March 1–21, 2026
- Future outcome window: March 22–31, 2026

## Excluded Data

Records with missing `content_hash_id` were excluded.

No client names, domains, URLs, private search queries, credentials, or raw exports are included in the analysis.

## Dataset Size

The selected March 2026 release contains 9,841,378 daily content-performance records before content-level aggregation.

#Import Libraries

In [93]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from huggingface_hub import login, hf_hub_download
from google.colab import userdata


#Login to Hugging Face

In [94]:
login(userdata.get("HF_TOKEN"))

#Load Dataset

In [95]:
parquet_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

con = duckdb.connect()

df = con.execute(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions,
    ga4_users,
    ga4_engaged_sessions,
    ga4_total_engagement_sec
FROM read_parquet('{parquet_file}')
""").fetch_df()

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 11)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,<NA>,<NA>,<NA>


#Verify Date Range

In [96]:
print("Start Date :", df["report_date"].min())
print("End Date   :", df["report_date"].max())

Start Date : 2026-03-01 00:00:00
End Date   : 2026-03-31 00:00:00


#Missing Values

In [97]:
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing[missing > 0]

,0
gsc_avg_position,6230317
ga4_pageviews,3018741
ga4_users,3018741
ga4_engaged_sessions,3018741
ga4_total_engagement_sec,3018741
ga4_sessions,3018741


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

# Methodology

## Features

Features are calculated only from the historical period (March 1–21, 2026):

- Past impressions
- Past clicks
- Past average position
- Past pageviews
- Past sessions
- Past users
- Past engaged sessions
- Past engagement time
- Past CTR

## Label

A content item is labeled as a future opportunity when its total clicks during the future period (March 22–31, 2026) are at or above the 90th percentile of future clicks.

For this dataset, the threshold is 1 future click.

## Baseline

The baseline is a majority-class classifier that predicts the most frequent class.

## Model

Random Forest Classifier with class-balanced weighting.

## Validation Design

Historical features are constructed from March 1–21, while the target is measured from March 22–31. The model is then evaluated on a held-out validation subset of the content-level dataset.

## Leakage Checks

- Future performance is not included in model features.
- Features are constructed only from the historical period.
- Validation predictions are generated only for content held out from model training.
- The final ranking evaluation uses the held-out validation set.

#Create CTR

In [98]:
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

df[["gsc_clicks", "gsc_impressions", "ctr"]].head()

,gsc_clicks,gsc_impressions,ctr
0,0,20,0.000
1,0,1,0.000
2,1,125,0.008
3,0,7,0.000
4,0,11,0.000


#Basic Cleaning

In [ ]:
selected_columns = [
    "report_date",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "ctr"
]

df = df[selected_columns].copy()

df = df.drop_duplicates()

df = df.dropna(subset=["content_hash_id"])

print(df.shape)

#Aggregate by Content

In [ ]:
content_df = (
    df.groupby("content_hash_id")
      .agg({
          "gsc_impressions": "sum",
          "gsc_clicks": "sum",
          "gsc_avg_position": "mean",
          "ga4_pageviews": "sum",
          "ga4_sessions": "sum",
          "ga4_users": "sum",
          "ga4_engaged_sessions": "sum",
          "ga4_total_engagement_sec": "sum",
          "ctr": "mean"
      })
      .reset_index()
)

print(content_df.shape)

content_df.head()

#Baseline Ranking

In [ ]:
baseline = (
    content_df.sort_values(
        by="gsc_impressions",
        ascending=False
    )
)

baseline.head(10)

#Create a Simple Content Opportunity Score

In [ ]:
content_df["opportunity_score"] = (
    0.40 * (1 - content_df["ctr"]) +
    0.30 * (
        content_df["gsc_impressions"] /
        content_df["gsc_impressions"].max()
    ) +
    0.30 * (
        content_df["gsc_avg_position"] /
        content_df["gsc_avg_position"].max()
    )
)

#Top Content Refresh Opportunities

In [ ]:
recommendations = (
    content_df
    .sort_values(
        by="opportunity_score",
        ascending=False
    )
    .head(20)
)

recommendations

In [ ]:
df.isnull().sum()

In [ ]:
numeric_cols = [
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec"
]

df[numeric_cols] = df[numeric_cols].fillna(0)

In [ ]:
df["gsc_avg_position"] = df["gsc_avg_position"].fillna(
    df["gsc_avg_position"].median()
)

In [ ]:
df = df.dropna(subset=["content_hash_id", "report_date"])

In [ ]:
df.isnull().sum()

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

# Results

The proposed model was compared against a simple baseline that ranks pages using impressions only.

#Compare Top 10 Rankings

In [ ]:
baseline_top10 = (
    content_df
    .sort_values("gsc_impressions", ascending=False)
    .head(10)
)

opportunity_top10 = (
    content_df
    .sort_values("opportunity_score", ascending=False)
    .head(10)
)

print("Top 10 Baseline")
display(baseline_top10)

print("\nTop 10 Opportunity Score")
display(opportunity_top10)

#Distribution of Opportunity Score

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(content_df["opportunity_score"], bins=30)

plt.title("Distribution of Content Opportunity Score")
plt.xlabel("Opportunity Score")
plt.ylabel("Number of Pages")

plt.show()

#CTR Distribution

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(content_df["ctr"], bins=10)

plt.title("CTR Distribution")
plt.xlabel("CTR")
plt.ylabel("Number of Pages")

plt.show()

#Impressions vs CTR

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    content_df["gsc_impressions"],
    content_df["ctr"],
    alpha=0.5
)

plt.title("CTR vs Impressions")
plt.xlabel("Impressions")
plt.ylabel("CTR")

plt.show()

#Top 20 Opportunity Pages

In [ ]:
top20 = (
    content_df
    .sort_values("opportunity_score", ascending=False)
    .head(20)
)

plt.figure(figsize=(12,6))

plt.bar(
    range(len(top20)),
    top20["opportunity_score"]
)

plt.xticks(
    range(len(top20)),
    top20["content_hash_id"],
    rotation=90,
    fontsize=8
)

plt.title("Top 20 Content Refresh Opportunities")
plt.xlabel("Content")
plt.ylabel("Opportunity Score")

plt.tight_layout()

plt.show()

#Correlation Heatmap

In [ ]:
import matplotlib.pyplot as plt

cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "ctr",
    "opportunity_score"
]

corr = content_df[cols].corr()

plt.figure(figsize=(10,8))

plt.imshow(corr)

plt.xticks(range(len(cols)), cols, rotation=90)
plt.yticks(range(len(cols)), cols)

plt.colorbar()

plt.title("Feature Correlation Matrix")

plt.show()

#Results Summary

## Summary

The baseline prioritizes content solely based on search impressions. In contrast, the proposed Content Opportunity Score incorporates multiple search performance signals, enabling a more comprehensive ranking of pages that may benefit from review.

Pages with high opportunity scores generally combine high visibility with relatively lower click-through rates or weaker average positions, making them suitable candidates for content refresh.

These findings are intended to support decision-making and should be interpreted as observational rather than causal.

## 5. Limitations

*What this work cannot claim.*

# Limitations

- Results are observational.
- Correlation does not imply causation.
- The model supports decision-making but does not explain Google's ranking algorithm.
- Only available warehouse data was analyzed.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

# 6. Ranked Recommendations

Based on the Content Opportunity Score, the following actions are recommended.

1. Prioritize pages with high impressions but low CTR for title and meta description improvements.
2. Review pages with declining average search position to identify content gaps.
3. Refresh pages with strong visibility but low engagement.
4. Continue monitoring high-performing pages to maintain performance.
5. Re-evaluate pages with consistently low visibility before investing additional resources.

These recommendations are intended to support content strategy and should be validated by domain experts before implementation.

In [ ]:
recommendations = (
    content_df
    .sort_values("opportunity_score", ascending=False)
    .head(20)
    .copy()
)

recommendations["Recommended Action"] = "Refresh Content"

recommendations[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "opportunity_score",
        "Recommended Action"
    ]
]

#Top 20 Recommendations

In [ ]:
recommendations = (
    content_df
    .sort_values("opportunity_score", ascending=False)
    .head(20)
    .copy()
)

recommendations["Recommended Action"] = "Refresh Content"

recommendations[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "opportunity_score",
        "Recommended Action"
    ]
]

In [ ]:
recommendations.to_csv(
    "top20_content_recommendations.csv",
    index=False
)

print("Top 20 recommendations saved successfully.")

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

# Charts and Tables

- Dataset overview
- Missing values
- Feature distributions
- Correlation heatmap
- Feature importance
- Model comparison table
- Top 20 content opportunities

#Save Figures

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(content_df["opportunity_score"], bins=30)

plt.title("Opportunity Score Distribution")

plt.savefig(
    "opportunity_score_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    content_df["gsc_impressions"],
    content_df["ctr"],
    alpha=0.5
)

plt.xlabel("Impressions")
plt.ylabel("CTR")
plt.title("Impressions vs CTR")

plt.savefig(
    "impressions_vs_ctr.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(10,8))

corr = content_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "ga4_users",
        "ga4_engaged_sessions",
        "ga4_total_engagement_sec",
        "ctr",
        "opportunity_score"
    ]
].corr()

plt.imshow(corr)

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.colorbar()

plt.title("Correlation Matrix")

plt.savefig(
    "correlation_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
content_df["gsc_avg_position"] = content_df["gsc_avg_position"].fillna(
    content_df["gsc_avg_position"].median()
)

In [ ]:
print(df["report_date"].min())
print(df["report_date"].max())

print(
    df["report_date"]
    .sort_values()
    .unique()
)

In [ ]:
df["report_date"] = pd.to_datetime(df["report_date"])

df = df.sort_values("report_date")

print(df["report_date"].min())
print(df["report_date"].max())

In [ ]:
cutoff_date = pd.Timestamp("2026-03-21")

past_df = df[df["report_date"] <= cutoff_date].copy()
future_df = df[df["report_date"] > cutoff_date].copy()

print("Past data:", past_df.shape)
print("Future data:", future_df.shape)

print("Past:", past_df["report_date"].min(), "to", past_df["report_date"].max())
print("Future:", future_df["report_date"].min(), "to", future_df["report_date"].max())

In [ ]:
past_features = (
    past_df.groupby("content_hash_id")
    .agg(
        past_impressions=("gsc_impressions", "sum"),
        past_clicks=("gsc_clicks", "sum"),
        past_avg_position=("gsc_avg_position", "mean"),
        past_pageviews=("ga4_pageviews", "sum"),
        past_sessions=("ga4_sessions", "sum"),
        past_users=("ga4_users", "sum"),
        past_engaged_sessions=("ga4_engaged_sessions", "sum"),
        past_engagement_sec=("ga4_total_engagement_sec", "sum")
    )
    .reset_index()
)

In [ ]:
past_features["past_ctr"] = np.where(
    past_features["past_impressions"] > 0,
    past_features["past_clicks"] / past_features["past_impressions"],
    0
)

In [ ]:
past_features.head()

In [ ]:
future_target = (
    future_df.groupby("content_hash_id")
    .agg(
        future_impressions=("gsc_impressions", "sum"),
        future_clicks=("gsc_clicks", "sum"),
        future_avg_position=("gsc_avg_position", "mean"),
        future_pageviews=("ga4_pageviews", "sum"),
        future_sessions=("ga4_sessions", "sum"),
        future_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
    .reset_index()
)

#Future Performance Metrics

In [ ]:
future_target["future_ctr"] = np.where(
    future_target["future_impressions"] > 0,
    future_target["future_clicks"] /
    future_target["future_impressions"],
    0
)

#Create Modeling Dataset

In [ ]:
model_df = past_features.merge(
    future_target,
    on="content_hash_id",
    how="inner"
)

print(model_df.shape)
model_df.head()

# Define Future Opportunity Threshold

In [ ]:
threshold = 1.0

print("Opportunity threshold:", threshold)

#Create Opportunity Label

In [ ]:
model_df["opportunity_label"] = (
    model_df["future_clicks"] >= threshold
).astype(int)

print(
    model_df["opportunity_label"].value_counts()
)

print(model_df["future_clicks"].describe())

#Define Model Features and Target

In [ ]:
X = model_df[
    [
        "past_impressions",
        "past_clicks",
        "past_avg_position",
        "past_pageviews",
        "past_sessions",
        "past_users",
        "past_engaged_sessions",
        "past_engagement_sec",
        "past_ctr"
    ]
]

y = model_df["opportunity_label"]

print("X:", X.shape)
print("y:", y.shape)

#Inspect Opportunity Distribution

In [ ]:
print(model_df.shape)
print(y.describe())

print(model_df["future_clicks"].describe())

print(
    model_df["future_clicks"]
    .value_counts()
    .sort_index()
    .head(20)
)

#Inspect Future Click Distribution

In [ ]:
print(
    model_df["future_clicks"]
    .value_counts()
    .sort_index()
    .head(20)
)

#Prepare Model Features and Target

In [ ]:
features = [
    "past_impressions",
    "past_clicks",
    "past_avg_position",
    "past_pageviews",
    "past_sessions",
    "past_users",
    "past_engaged_sessions",
    "past_engagement_sec",
    "past_ctr"
]

X = model_df[features].copy()
y = model_df["opportunity_label"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts())

#Train Majority-Class Baseline

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X, y)

baseline_pred = baseline.predict(X)

print("Baseline positive predictions:",
      baseline_pred.sum())

#Create Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

# Train Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train, y_train)

model_pred = rf.predict(X_val)
model_prob = rf.predict_proba(X_val)[:, 1]

#Train Majority-Class Baseline

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_val)
baseline_prob = baseline.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_model(name, y_true, pred, prob):

    print(f"\n{name}")
    print("-" * 40)

    print("Accuracy :", accuracy_score(y_true, pred))
    print("Precision:", precision_score(y_true, pred, zero_division=0))
    print("Recall   :", recall_score(y_true, pred, zero_division=0))
    print("F1       :", f1_score(y_true, pred, zero_division=0))
    print("ROC-AUC  :", roc_auc_score(y_true, prob))
    print("PR-AUC   :", average_precision_score(y_true, prob))


evaluate_model(
    "Baseline",
    y_val,
    baseline_pred,
    baseline_prob
)

evaluate_model(
    "Random Forest",
    y_val,
    model_pred,
    model_prob
)

#Random Forest Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_val,
    rf_pred
)

plt.title("Random Forest Confusion Matrix")
plt.show()

# Generate Opportunity Probabilities

In [ ]:
# Predict probability of future opportunity on validation data
val_results = model_df.loc[X_val.index].copy()

val_results["opportunity_probability"] = rf.predict_proba(X_val)[:, 1]

val_results[
    [
        "content_hash_id",
        "opportunity_probability",
        "past_impressions",
        "past_clicks",
        "past_ctr"
    ]
].head()

#Rank Top 20 Content Opportunities

In [ ]:
top20_recommendations = (
    val_results
    .sort_values(
        "opportunity_probability",
        ascending=False
    )
    .head(20)
    .copy()
)

top20_recommendations[
    [
        "content_hash_id",
        "opportunity_probability",
        "past_impressions",
        "past_clicks",
        "past_ctr",
        "past_avg_position"
    ]
]

#Generate Recommendation Reason Codes

In [ ]:
# Calculate historical medians once
impressions_median = model_df["past_impressions"].median()
ctr_median = model_df["past_ctr"].median()
position_median = model_df["past_avg_position"].median()


def get_reason(row):

    reasons = []

    if row["past_impressions"] > impressions_median:
        reasons.append("high visibility")

    if row["past_ctr"] < ctr_median:
        reasons.append("low CTR")

    if row["past_avg_position"] > position_median:
        reasons.append("weaker average position")

    if row["past_clicks"] == 0:
        reasons.append("no historical clicks")

    if not reasons:
        reasons.append("model-ranked opportunity")

    return ", ".join(reasons)


top20_recommendations["reason"] = (
    top20_recommendations.apply(
        get_reason,
        axis=1
    )
)

#Create Final Ranked Recommendations

In [ ]:
final_recommendations = top20_recommendations[
    [
        "content_hash_id",
        "opportunity_probability",
        "past_impressions",
        "past_clicks",
        "past_ctr",
        "past_avg_position",
        "reason"
    ]
].copy()

final_recommendations.index = range(1, len(final_recommendations) + 1)

display(final_recommendations)

#Export Ranked Recommendations

In [ ]:
final_recommendations.to_csv(
    "ranked_content_recommendations.csv"
    )

print("Recommendations saved successfully.")

#Visualize Top 20 Ranked Opportunities

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    final_recommendations["content_hash_id"].astype(str),
    final_recommendations["opportunity_probability"]
)

plt.xlabel("Predicted Opportunity Probability")
plt.ylabel("Content")
plt.title("Top 20 Ranked Content Opportunities")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Ranked Recommendations

The model was used to rank content by predicted probability of receiving future clicks.

The highest-ranked content represents pages that the model identifies as stronger candidates for review based on historical search and engagement signals.

Reason codes provide an interpretable explanation for each recommendation, such as high visibility, low CTR, or weaker average search position.

These rankings are directional and intended to support content-review prioritization. They do not imply that refreshing a page will cause future performance to improve.

# Validate Top 20 Recommendations

In [ ]:
top20_evidence = top20_recommendations[
    [
        "content_hash_id",
        "opportunity_probability",
        "past_impressions",
        "past_clicks",
        "past_ctr",
        "past_avg_position"
    ]
].merge(
    val_results[
        [
            "content_hash_id",
            "future_clicks",
            "future_impressions",
            "future_ctr"
        ]
    ],
    on="content_hash_id",
    how="left"
)

display(top20_evidence)

#Validate Top 20 Opportunity Rate

In [ ]:
top20_evidence["actual_opportunity"] = (
    top20_evidence["future_clicks"] >= threshold
).astype(int)

print(
    "Actual opportunities in Top 20:",
    top20_evidence["actual_opportunity"].sum(),
    "out of",
    len(top20_evidence)
)

#Calculate Precision@20

In [ ]:
precision_at_20 = (
    top20_evidence["actual_opportunity"].sum()
    / len(top20_evidence)
)

print(f"Precision@20: {precision_at_20:.3f}")

#Compare Against Random Selection

In [ ]:
random_sample = val_results.sample(
    n=20,
    random_state=42
).copy()

random_sample["actual_opportunity"] = (
    random_sample["future_clicks"] >= threshold
).astype(int)

random_precision_at_20 = (
    random_sample["actual_opportunity"].sum()
    / len(random_sample)
)

print(
    f"Random Precision@20: {random_precision_at_20:.3f}"
)

#Compare Ranking Strategies

In [ ]:
ranking_comparison = pd.DataFrame({
    "Strategy": [
        "Random selection",
        "Random Forest ranking"
    ],
    "Precision@20": [
        random_precision_at_20,
        precision_at_20
    ]
})

display(ranking_comparison)

#Create Final Paper Results Table

In [ ]:
final_paper_table = top20_evidence[
    [
        "content_hash_id",
        "opportunity_probability",
        "past_impressions",
        "past_clicks",
        "past_ctr",
        "past_avg_position",
        "future_clicks",
        "future_ctr",
        "actual_opportunity"
    ]
].copy()

final_paper_table.index = range(
    1,
    len(final_paper_table) + 1
)

display(final_paper_table)

#Rank Validation Opportunities by Predicted Probability

In [ ]:
# Validation set probabilities
val_results = model_df.loc[X_val.index].copy()

val_results["opportunity_probability"] = (
    rf.predict_proba(X_val)[:, 1]
)

val_results = val_results.sort_values(
    "opportunity_probability",
    ascending=False
)

display(
    val_results[
        [
            "content_hash_id",
            "opportunity_probability",
            "opportunity_label",
            "future_clicks"
        ]
    ].head(20)
)

#Calculate Validation Precision@20

In [ ]:
top20_val = val_results.head(20).copy()

precision_at_20_val = (
    top20_val["opportunity_label"].sum()
    / len(top20_val)
)

print(f"Validation Precision@20: {precision_at_20_val:.3f}")
print(
    "Actual opportunities:",
    top20_val["opportunity_label"].sum(),
    "out of",
    len(top20_val)
)

#Compare Against Random Validation Selection

In [ ]:
random_val = model_df.loc[X_val.index].sample(
    n=20,
    random_state=42
).copy()

random_precision_at_20 = (
    random_val["opportunity_label"].sum()
    / len(random_val)
)

print(f"Random Precision@20: {random_precision_at_20:.3f}")

#Compare Ranking Strategies

In [ ]:
ranking_results = pd.DataFrame({
    "Strategy": [
        "Random selection",
        "Random Forest ranking"
    ],
    "Precision@20": [
        random_precision_at_20,
        precision_at_20_val
    ]
})

display(ranking_results)

#Compare Model Performance with Baseline

In [ ]:
import matplotlib.pyplot as plt

metrics = [
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC"
]

baseline_scores = [
    precision_score(y_val, baseline_pred, zero_division=0),
    recall_score(y_val, baseline_pred, zero_division=0),
    f1_score(y_val, baseline_pred, zero_division=0),
    roc_auc_score(y_val, baseline_prob),
    average_precision_score(y_val, baseline_prob)
]

rf_scores = [
    precision_score(y_val, rf_pred, zero_division=0),
    recall_score(y_val, rf_pred, zero_division=0),
    f1_score(y_val, rf_pred, zero_division=0),
    roc_auc_score(y_val, rf_prob),
    average_precision_score(y_val, rf_prob)
]

x = range(len(metrics))

plt.figure(figsize=(10, 6))

plt.plot(x, baseline_scores, marker="o", label="Baseline")
plt.plot(x, rf_scores, marker="o", label="Random Forest")

plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("Random Forest vs Majority Baseline")
plt.legend()

plt.tight_layout()
plt.show()

# Feature Importance

The Random Forest feature importances are used to summarize which historical features contributed most to the model's predictions. These are model-derived associations and should not be interpreted as causal effects.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(feature_importance)

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    feature_importance["feature"],
    feature_importance["importance"]
)

plt.xlabel("Random Forest Feature Importance")
plt.ylabel("Feature")
plt.title("Historical Feature Importance")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Limitations and Honest Framing

This analysis is based on observed FlyRank search and engagement data from the March 2026 analysis window.

The model identifies directional patterns associated with future opportunity; it does not establish causation or prove that changing a content item will produce a specific search outcome.

The validation period is limited to the available future window, so performance may differ on other periods or datasets.

Precision@20 is based on one held-out validation set and should not be treated as a universal performance guarantee.

The random comparison uses one fixed-seed sample and is provided as a simple reference rather than a definitive estimate of random-selection performance.

Recommendations are intended for decision support and content review prioritization, not automated publishing decisions.

# Artifacts and Reproducibility

The notebook produces the following artifacts for the research paper:

- Model-versus-baseline performance chart
- Random Forest confusion matrix
- Historical feature importance chart
- Top-20 ranked opportunity table
- Precision@20 comparison
- Ranked content recommendations CSV

The analysis uses the same held-out validation split for baseline and Random Forest evaluation.

The recommendation ranking is based on predicted opportunity probability, while future-period outcomes are used only for validation and evaluation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.